# Notebook 01 — Explore Landsat NDVI

Goal: fetch annual NDVI for a single coordinate and understand the data quality, cloud masking, and Landsat generation gaps before putting it in production.

In [ ]:
import sys; sys.path.insert(0, '..')
import ee
import pandas as pd
import matplotlib.pyplot as plt

# Authenticate — use service account in prod, interactive here
ee.Authenticate()
ee.Initialize(project='your-gcp-project')

In [ ]:
# Test coordinate: Bangalore urban core
LAT, LON = 12.97, 77.59
region = ee.Geometry.Point([LON, LAT]).buffer(500)

from apps.api.services.gee_fetcher import _fetch_ndvi
ndvi_series = _fetch_ndvi(region)
df = pd.DataFrame(ndvi_series)
df.head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
valid = df.dropna(subset=['value'])
ax.plot(valid['year'], valid['value'], 'o-', color='#22d3ee', lw=1.5, ms=4)
ax.axvline(2013, color='orange', alpha=0.5, linestyle='--', label='L7→L8')
ax.axvline(2021, color='red', alpha=0.5, linestyle='--', label='L8→L9')
ax.set_xlabel('Year'); ax.set_ylabel('NDVI'); ax.legend()
ax.set_title(f'Annual NDVI — Bangalore ({LAT}, {LON})')
plt.tight_layout()
plt.show()

## Notes on data gaps

- Years with `null` values are typically high cloud-cover seasons or Landsat sensor transitions.
- The 2003 SLC-off failure on Landsat 7 causes partial data — use L5 through 2012 as primary.
- Growing season filter (May–Sep) reduces cloud interference for most mid-latitudes; adjust for tropical regions.

In [ ]:
# Check null rate
null_rate = df['value'].isna().mean()
print(f'Null rate: {null_rate:.1%}')
print(f'Years with data: {df["value"].notna().sum()} / {len(df)}')